#36615 - Final Project Part 3 - Delay Prediction Spark ML
## Team Polyhymnia
### Vraj Patel, Jay Louissant, Elaine Yin

Goal: Use Spark ML with engineered features to predict
whether a flight will be delayed (including cancellations).

In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number # Import from functions

from pyspark.ml.feature import RFormula, StandardScaler
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder
import pandas as pd

from pyspark.ml.classification import RandomForestClassifier, RandomForestClassificationModel
from pyspark.ml import Pipeline, PipelineModel

#### 1. Load feature-engineered data

In [0]:
df = spark.table("lsd_2026.default.polyhymnia_feature_engineered")

print(f"Rows: {df.count():,}")
df.printSchema()

#### 2. Define target label: departure delayed or not

In [0]:
df = df.withColumn(
    "dep_delayed",
    F.when((F.col("DEP_DELAY") > 0) | (F.col("CANCELLED") == 1), F.lit(1)).otherwise(F.lit(0))
)

df.groupBy("dep_delayed").count().show()

We see that about 37% of flights in the dataset (over the decade) were delayed.

#### 3. Train / Test / Validation split strategy

Because we used rolling/time-based features (previous hour, past 7 days), we will not perform a fully random split. Instead, we will sort by FL_DATE (and CRS_DEP_TIME) and use the earliest 60% as the training set and next 20% as test set (for model selection / tuning). Final 20% will be used as final validation (held out).


In [0]:
import pyspark.sql.functions as F

# create a numeric representation of time for splitting
# combining date and time into a single numeric value
df = df.withColumn("time_rank_col", 
                   F.unix_timestamp(F.to_date("FL_DATE", "yyyy-MM-dd")) + (F.col("CRS_DEP_TIME").cast("int") * 60))

# find the 60th and 80th percentile values
# 0.001 is the relative error: lower is more accurate but slower
split_points = df.stat.approxQuantile("time_rank_col", [0.6, 0.8], 0.001)

train_threshold = split_points[0]
test_threshold = split_points[1]

# perform the split
train = df.filter(F.col("time_rank_col") <= train_threshold)
test  = df.filter((F.col("time_rank_col") > train_threshold) & (F.col("time_rank_col") <= test_threshold))
valid = df.filter(F.col("time_rank_col") > test_threshold)

# cache the splits so Spark doesn't re-calculate them for every model in the grid
train.cache()
test.cache()

#### 4. Create an Rformula to generate model matrix

In [0]:
formula_str = """
dep_delayed ~ 
    OP_CARRIER + ORIGIN + DEST +
    CRS_DEP_TIME + CRS_ARR_TIME + CRS_ELAPSED_TIME + DISTANCE +
    day_of_week + hour_of_day +
    crs_dep_time_bucket + arr_time_bucket +
    rate_weather_delay_dep + rate_weather_delay_arr +
    dep_traffic_z_score + carrier_delay_rate_7d + route_delay_rate_7d
"""

rf_stage = RFormula(
    formula=formula_str,
    featuresCol="raw_features",
    labelCol="label",
    handleInvalid="skip"
)

#### 5. Apply the Rformula to generate model matrices for the data

In [0]:
rf_model = rf_stage.fit(train)

train_rf = rf_model.transform(train)
test_rf  = rf_model.transform(test)
valid_rf = rf_model.transform(valid)


#### 6. Standardize features

Note, we standardize only using the training data, and then apply the mean and standard deviation thereby derived to the validation and test sets to standardize them.

In [0]:
# fit Scaler on train_rf only
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
scaler_model = scaler.fit(train_rf)

# apply the learned mean/std to all sets
train_scaled = scaler_model.transform(train_rf)
test_scaled  = scaler_model.transform(test_rf)
valid_scaled = scaler_model.transform(valid_rf)

# pre-fetch columns to speed up the loop
train_scaled = train_scaled.select("label", "features").cache()
test_scaled = test_scaled.select("label", "features").cache()

#### 7. Random Forest classifier pipeline - fit, tune, evaluate 

In [0]:
rf = RandomForestClassifier(labelCol="label", featuresCol="features")

paramGrid = ParamGridBuilder() \
    .addGrid(rf.maxDepth, [2, 3, 5]) \
    .addGrid(rf.numTrees, [10, 15, 20]) \
    .build()

evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")

best_acc = 0.0
best_params = None

# grid Search
for params in paramGrid:
    # set the parameters for this run
    rf_curr = rf.copy(params)
    print(f"Starting Fit: {params}")
    
    model = rf_curr.fit(train_scaled)
    preds = model.transform(test_scaled)
    acc = evaluator.evaluate(preds)
    
    print(f"Resulting Accuracy: {acc}")
    if acc > best_acc:
        best_acc = acc
        best_params = params
        # save the current best model immediately in case of crash
        model.write().overwrite().save("lsd_2026.default.polyhymnia_best_rf_model_checkpoint")

print(f"Optimal Parameters: {best_params}")

#### 8. Refit Decision Tree with best parameters on train + test, evaluate on Validation

In [0]:
from pyspark.ml import Pipeline

# use best parameters found (example: depth 5, trees 75)
final_rf = RandomForestClassifier(labelCol="label", featuresCol="features", 
                                  maxDepth=best_params[rf.maxDepth], 
                                  numTrees=best_params[rf.numTrees])

# combine Train and Test for final training
final_train_data = train_scaled.union(test_scaled)

# final fit
final_rf_model = final_rf.fit(final_train_data)

# create a final PipelineModel to save everything (Formula -> Scaler -> RF)
final_pipeline = PipelineModel(stages=[rf_model, scaler_model, final_rf_model])

# save to a persistent location
final_pipeline.write().overwrite().save("lsd_2026.default.polyhymnia_delay_prediction_model_RF")

#### 9. Confusion matrix derived metrics on held out validation set

In [0]:
confusion_matrix = (
    valid_preds
    .groupBy("label", "prediction")
    .count()
    .toPandas()
)

print("Confusion matrix (validation):")
print(confusion_matrix)

TP = confusion_matrix[(confusion_matrix["label"] == 1.0) & (confusion_matrix["prediction"] == 1.0)]["count"].sum()
TN = confusion_matrix[(confusion_matrix["label"] == 0.0) & (confusion_matrix["prediction"] == 0.0)]["count"].sum()
FP = confusion_matrix[(confusion_matrix["label"] == 0.0) & (confusion_matrix["prediction"] == 1.0)]["count"].sum()
FN = confusion_matrix[(confusion_matrix["label"] == 1.0) & (confusion_matrix["prediction"] == 0.0)]["count"].sum()

TP, TN, FP, FN = map(float, [TP, TN, FP, FN])

tpr = TP / (TP + FN) if (TP + FN) > 0 else 0.0  # sensitivity / recall
fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0

print(f"\nValidation metrics (positive class = delayed):")
print(f"Accuracy:    {rf_valid_accuracy:.4f}")
print(f"TPR / Sensitivity: {tpr:.4f}")
print(f"FPR:         {fpr:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1 Score:    {rf_valid_f1:.4f}")


#### 10. Metrics for baseline model - always predict "no delay"

In [0]:
baseline = valid_scaled.withColumn("prediction", F.lit(0.0))

baseline_matrix = (
    baseline
    .groupBy("label", "prediction")
    .count()
    .toPandas()
)

print("Baseline confusion matrix (always predict no delay):")
print(baseline_matrix)

B_TP = baseline_matrix[(baseline_matrix["label"] == 1.0) & (baseline_matrix["prediction"] == 1.0)]["count"].sum()
B_TN = baseline_matrix[(baseline_matrix["label"] == 0.0) & (baseline_matrix["prediction"] == 0.0)]["count"].sum()
B_FP = baseline_matrix[(baseline_matrix["label"] == 0.0) & (baseline_matrix["prediction"] == 1.0)]["count"].sum()
B_FN = baseline_matrix[(baseline_matrix["label"] == 1.0) & (baseline_matrix["prediction"] == 0.0)]["count"].sum()

B_TP, B_TN, B_FP, B_FN = map(float, [B_TP, B_TN, B_FP, B_FN])

B_accuracy = (B_TP + B_TN) / (B_TP + B_TN + B_FP + B_FN) if (B_TP + B_TN + B_FP + B_FN) > 0 else 0.0
B_tpr = B_TP / (B_TP + B_FN) if (B_TP + B_FN) > 0 else 0.0
B_fpr = B_FP / (B_FP + B_TN) if (B_FP + B_TN) > 0 else 0.0
B_specificity = B_TN / (B_TN + B_FP) if (B_TN + B_FP) > 0 else 0.0

print(f"\nBaseline metrics (always predict no delay):")
print(f"Accuracy:    {B_accuracy:.4f}")
print(f"TPR / Sensitivity: {B_tpr:.4f}")
print(f"FPR:         {B_fpr:.4f}")
print(f"Specificity: {B_specificity:.4f}")


#### 11. Comparison Summary

In [0]:
print("\n=== Validation Performance Comparison (Best DT vs Baseline) ===")
print(f"Decision Tree Accuracy vs Baseline: {dt_valid_accuracy:.4f} vs {B_accuracy:.4f}")
print(f"Decision Tree TPR vs Baseline:      {tpr:.4f} vs {B_tpr:.4f}")
print(f"Decision Tree FPR vs Baseline:      {fpr:.4f} vs {B_fpr:.4f}")
print(f"Decision Tree Specificity vs Base:  {specificity:.4f} vs {B_specificity:.4f}")
